# 1st try

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
 
%config InlineBackend.figure_format = 'retina'
 
import matplotlib.font_manager as fm
fontpath = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
font = fm.FontProperties(fname=fontpath, size=9)
plt.rc('font', family='NanumBarunGothic') 
mpl.font_manager.findfont(font)

print("완료!")

완료!


In [2]:
import tensorflow as tf
import numpy as np

from sklearn.model_selection import train_test_split

import matplotlib.ticker as ticker
import matplotlib.pyplot as plt

import time
import re
import os
import io

print(tf.__version__)

2.6.0


In [8]:
# 파일 경로 설정
ko_path = '/aiffel/aiffel/s2s_translation/korean-english-park.train.ko'
en_path = '/aiffel/aiffel/s2s_translation/korean-english-park.train.en'

# 파일 로드
with open(ko_path, "r", encoding="utf-8") as f:
    korean_sentences = f.readlines()

with open(en_path, "r", encoding="utf-8") as f:
    english_sentences = f.readlines()

# 데이터 개수 확인
print(f"한국어 문장 개수: {len(korean_sentences)}")
print(f"영어 문장 개수: {len(english_sentences)}")

# 첫 5개 샘플 출력
for i in range(5):
    print(f"KO: {korean_sentences[i].strip()}")
    print(f"EN: {english_sentences[i].strip()}")
    print()


한국어 문장 개수: 94123
영어 문장 개수: 94123
KO: 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
EN: Much of personal computing is about "can you top this?"

KO: 모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하지 않는다.
EN: so a mention a few weeks ago about a rechargeable wireless optical mouse brought in another rechargeable, wireless mouse.

KO: 그러나 이것은 또한 책상도 필요로 하지 않는다.
EN: Like all optical mice, But it also doesn't need a desk.

KO: 79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분이든 그 움직임에따라 커서의 움직임을 조절하는 회전 운동 센서를 사용하고 있다.
EN: uses gyroscopic sensors to control the cursor movement as you move your wrist, arm, whatever through the air.

KO: 정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔음을 밝혔으며, 세계 해상 교역량의 거의 3분의 1을 운송하는 좁은 해로인 말라카 해협이 테러 공격을 당하기 쉽다고 경고하고 있다.
EN: Intelligence officials have revealed a spate of foiled plots on ships in Southeast Asia and are warning that a narrow stretch of water carrying almost one third of the world's maritime trade is vulnerable to a terror attack.



In [14]:
import re
from konlpy.tag import Mecab

# Mecab 형태소 분석기 로드
mecab = Mecab()

def preprocess_english(sentence):
    """영어 문장 전처리 함수"""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    sentence = sentence.strip()
    sentence = "<start> " + sentence + " <end>"
    return sentence

def preprocess_korean(sentence):
    """한글 문장 전처리 및 형태소 분석"""
    sentence = sentence.strip()
    sentence = mecab.morphs(sentence)  # 형태소 분석
    sentence = " ".join(sentence)  # 토큰을 공백으로 결합
    return sentence

# ✅ 순서를 유지한 중복 제거 (set 대신 리스트 사용)
seen = set()
cleaned_corpus = []
for ko, en in zip(korean_sentences, english_sentences):
    ko_cleaned = preprocess_korean(ko.strip())
    en_cleaned = preprocess_english(en.strip())
    if (ko_cleaned, en_cleaned) not in seen:
        seen.add((ko_cleaned, en_cleaned))
        cleaned_corpus.append((ko_cleaned, en_cleaned))

# ✅ 길이 필터링 (40 토큰 이하)
filtered_corpus = [(ko, en) for ko, en in cleaned_corpus if len(ko.split()) <= 40 and len(en.split()) <= 40]

# 필터링 후 데이터 개수 출력
print(f"정제 후 데이터 개수: {len(filtered_corpus)}")

# 상위 10개 샘플 확인
for i in range(10):
    print(f"{i+1}. KO: {filtered_corpus[i][0]}")
    print(f"   EN: {filtered_corpus[i][1]}")
    print()


정제 후 데이터 개수: 59754
1. KO: 개인 용 컴퓨터 사용 의 상당 부분 은 " 이것 보다 뛰어날 수 있 느냐 ? "
   EN: <start> much of personal computing is about can you top this ? <end>

2. KO: 모든 광 마우스 와 마찬가지 로 이 광 마우스 도 책상 위 에 놓 는 마우스 패드 를 필요 로 하 지 않 는다 .
   EN: <start> so a mention a few weeks ago about a rechargeable wireless optical mouse brought in another rechargeable , wireless mouse . <end>

3. KO: 그러나 이것 은 또한 책상 도 필요 로 하 지 않 는다 .
   EN: <start> like all optical mice , but it also doesn t need a desk . <end>

4. KO: " 결정 적 인 순간 에 그 들 의 능력 을 증가 시켜 줄 그 무엇 이 매우 중요 합니다 . "
   EN: <start> something that will boost their capabilities at crucial moments is very important . <end>

5. KO: 연구가 들 이 이미 커피 대체 품 으로서 음식 대용 과자 나 껌 에 카페인 을 첨가 하 는 방법 을 연구 하 고 있 다고 Archibald 는 말 했 다 .
   EN: <start> researchers are already exploring ways to put caffeine in nutrition bars or chewing gum as alternatives to coffee , archibald said . <end>

6. KO: 의약 연구소 는 정부 에 과학 문제 에 관해 자문 하 기 위해 의회 가 설립 인가 를 내 어 준 민간 단체 인 국립 과학 학회 의 부속 단체 이 다 .
   EN:

In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 최대 단어 개수 (최소 10,000 이상)
vocab_size = 20000  # 10,000보다 여유롭게 잡음
max_length = 40  # 최대 시퀀스 길이 (Step 2에서 필터링 기준)

# ✅ 1. 한국어 토크나이저 (Mecab 기반)
ko_tokenizer = Tokenizer(num_words=vocab_size, filters="", oov_token="<unk>")
ko_texts = [ko for ko, en in filtered_corpus]  # 한국어 문장 리스트
ko_tokenizer.fit_on_texts(ko_texts)

# ✅ 2. 영어 토크나이저 (<start>, <end> 포함)
en_tokenizer = Tokenizer(num_words=vocab_size, filters="", oov_token="<unk>")
en_texts = [en for ko, en in filtered_corpus]  # 영어 문장 리스트
en_tokenizer.fit_on_texts(en_texts)

# ✅ 3. 텍스트를 정수 시퀀스로 변환
encoder_input = ko_tokenizer.texts_to_sequences(ko_texts)
decoder_input = en_tokenizer.texts_to_sequences(en_texts)

# ✅ 4. 패딩 적용
encoder_input = pad_sequences(encoder_input, maxlen=max_length, padding="post")
decoder_input = pad_sequences(decoder_input, maxlen=max_length, padding="post")

# ✅ 5. 결과 확인
print(f"한국어 토큰화 샘플: {encoder_input[0]}")
print(f"영어 토큰화 샘플: {decoder_input[0]}")

# ✅ 6. 단어 사전 크기 확인
print(f"한국어 단어 사전 크기: {len(ko_tokenizer.word_index)}")
print(f"영어 단어 사전 크기: {len(en_tokenizer.word_index)}")


한국어 토큰화 샘플: [ 788  660  567  198    7 1419  893    8   60  724  184    1   45   15
 3807  303   60    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]
영어 토큰화 샘플: [   4  277    8 1333 7787   17   44   95   86  207   41  183    5    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]
한국어 단어 사전 크기: 44259
영어 단어 사전 크기: 37302


In [45]:
import tensorflow as tf

# 하이퍼파라미터 설정
embedding_dim = 256  # 임베딩 크기
units = 512  # LSTM hidden state 크기
vocab_size_ko = 20000  # 한국어 단어 사전 크기
vocab_size_en = 20000  # 영어 단어 사전 크기
max_length = 40  # 문장 최대 길이

# ✅ 인코더 (Encoder)
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(enc_units, return_sequences=True, return_state=True, dropout=0.3)

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, state_h, state_c

    def initialize_hidden_state(self, batch_size):
        return [tf.zeros((batch_size, self.enc_units)), tf.zeros((batch_size, self.enc_units))]

# ✅ Bahdanau Attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# ✅ 디코더 (Decoder)
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(dec_units, return_sequences=True, return_state=True, dropout=0.3)
        self.fc = tf.keras.layers.Dense(vocab_size, activation='softmax')
        self.attention = BahdanauAttention(dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden[0], enc_output)

        # ✅ 배치 크기 맞추기
        context_vector = tf.expand_dims(context_vector, 1)
        context_vector = tf.tile(context_vector, [1, tf.shape(x)[1], 1])

        x = self.embedding(x)  # (batch_size, 1, embedding_dim)
        x = tf.concat([context_vector, x], axis=-1)  # (batch_size, 1, hidden_size + embedding_dim)

        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)

        return x, state_h, state_c, attention_weights





In [46]:
optimizer = tf.keras.optimizers.Adam()

# ✅ 손실 함수 정의
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask  # 패딩 토큰 제외
    return tf.reduce_mean(loss_)

# ✅ 학습 Step 정의
@tf.function  # ✅ 다시 추가하여 속도 최적화
def train_step(inp, targ, enc_hidden):
    loss = 0
    batch_size = tf.shape(inp)[0]

    with tf.GradientTape() as tape:
        enc_output, enc_hidden_h, enc_hidden_c = encoder(inp, encoder.initialize_hidden_state(batch_size))
        dec_hidden = [enc_hidden_h, enc_hidden_c]

        # ✅ batch_size 반영하여 dec_input 생성
        dec_input = tf.expand_dims(tf.repeat([en_tokenizer.word_index['<start>']], batch_size), 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden_h, dec_hidden_c, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)
            dec_hidden = [dec_hidden_h, dec_hidden_c]

    batch_loss = loss / int(targ.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss



In [47]:
# ✅ 모델 인스턴스 생성
batch_size = 64
encoder = Encoder(vocab_size_ko, embedding_dim, units)
decoder = Decoder(vocab_size_en, embedding_dim, units)

from tqdm import tqdm

# ✅ 학습 루프 수정 (batch_size 전달)
num_epochs = 10  # 실험적으로 변경 가능

for epoch in range(num_epochs):
    total_loss = 0
    enc_hidden = encoder.initialize_hidden_state(batch_size)  # ← batch_size 전달

    print(f'Epoch {epoch + 1}/{num_epochs}')
    
    # tqdm 추가된 배치 진행률 표시
    for batch in tqdm(range(0, len(encoder_input), batch_size), desc=f"Training Epoch {epoch + 1}"):
        inp = encoder_input[batch:batch + batch_size]
        targ = decoder_input[batch:batch + batch_size]

        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

    print(f'Epoch {epoch + 1}, Loss: {total_loss.numpy()}')


Epoch 1/10


Training Epoch 1: 100%|██████████| 934/934 [04:21<00:00,  3.57it/s] 


Epoch 1, Loss: 3117.18017578125
Epoch 2/10


Training Epoch 2: 100%|██████████| 934/934 [03:06<00:00,  5.00it/s]


Epoch 2, Loss: 2691.333740234375
Epoch 3/10


...

Epoch 8, Loss: 1813.39453125
Epoch 9/10


Training Epoch 9: 100%|██████████| 934/934 [03:06<00:00,  5.00it/s]


Epoch 9, Loss: 1713.125
Epoch 10/10


Training Epoch 10: 100%|██████████| 934/934 [03:07<00:00,  4.99it/s]


Epoch 10, Loss: 1619.022705078125


In [55]:
# 모델 저장
encoder.save("encoder_model_20000_40")
decoder.save("decoder_model_20000_40")

# # 나중에 불러올 때
# encoder = tf.keras.models.load_model("encoder_model")
# decoder = tf.keras.models.load_model("decoder_model")


INFO:tensorflow:Assets written to: encoder_model_20000_40/assets


INFO:tensorflow:Assets written to: encoder_model_20000_40/assets


INFO:tensorflow:Assets written to: decoder_model_20000_40/assets


INFO:tensorflow:Assets written to: decoder_model_20000_40/assets


In [52]:
def evaluate(sentence):
    sentence = preprocess_korean(sentence)  # ✅ 한글 전처리 함수 사용
    inputs = ko_tokenizer.texts_to_sequences([sentence])
    inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=max_length, padding='post')

    enc_hidden = encoder.initialize_hidden_state(1)
    enc_output, enc_hidden_h, enc_hidden_c = encoder(tf.convert_to_tensor(inputs), enc_hidden)
    dec_hidden = [enc_hidden_h, enc_hidden_c]

    dec_input = tf.expand_dims([en_tokenizer.word_index['<start>']], 0)
    result = []

    for _ in range(max_length):
        predictions, dec_hidden_h, dec_hidden_c, _ = decoder(dec_input, dec_hidden, enc_output)
        predicted_id = tf.argmax(predictions[0]).numpy()

        if en_tokenizer.index_word[predicted_id] == '<end>':
            break

        result.append(en_tokenizer.index_word[predicted_id])
        dec_input = tf.expand_dims([predicted_id], 0)
        dec_hidden = [dec_hidden_h, dec_hidden_c]

    return ' '.join(result)


def translate(sentence):
    print(f'KO: {sentence}')
    result = evaluate(sentence)
    print(f'EN: {result}')


In [53]:
# 테스트 문장
test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

# 번역 실행
for sentence in test_sentences:
    translate(sentence)


KO: 오바마는 대통령이다.
EN: obama is expected to be a new president .
KO: 시민들은 도시 속에 산다.
EN: they are also waiting to the scene of the <unk> .
KO: 커피는 필요 없다.
EN: the company is not a very tough form .
KO: 일곱 명의 사망자가 발생했다.
EN: at least people were killed .


In [56]:
!pip install sacrebleu

     |████████████████████████████████| 104 kB 5.6 MB/s            


In [58]:
import random
import nltk
nltk.download('wordnet')
from nltk.translate.meteor_score import meteor_score
from sacrebleu.metrics import TER

# ✅ 테스트 데이터 로드
with open('/aiffel/aiffel/s2s_translation/korean-english-park.test.ko', 'r', encoding='utf-8') as f:
    test_ko_sentences = [line.strip() for line in f.readlines()]

with open('/aiffel/aiffel/s2s_translation/korean-english-park.test.en', 'r', encoding='utf-8') as f:
    test_en_sentences = [line.strip() for line in f.readlines()]

In [ ]:
# ✅ 모델 번역 실행
translated_sentences = []
for sentence in tqdm(test_ko_sentences, desc="Translating", unit="sentence"):
    translated_sentences.append(evaluate(sentence))
 

In [61]:
# ✅ 랜덤 시드 고정 함수
def set_random_seed(seed=42):
    random.seed(seed)

# ✅ METEOR 점수 계산 함수 (토큰화 추가)
def calculate_meteor(reference_texts, translated_texts):
    scores = [meteor_score([ref.split()], trans.split()) for ref, trans in zip(reference_texts, translated_texts)]
    return sum(scores) / len(scores), scores  # 전체 평균 점수, 개별 점수 리스트 반환

# ✅ TER 점수 계산 함수
def calculate_ter(reference_texts, translated_texts):
    ter = TER()
    score = ter.corpus_score(translated_texts, [reference_texts])
    individual_scores = [ter.sentence_score(trans, [ref]).score for trans, ref in zip(translated_texts, reference_texts)]
    return score.score, individual_scores  # 전체 평균 점수, 개별 점수 리스트 반환

# ✅ METEOR & TER 점수 계산
meteor_avg, meteor_scores = calculate_meteor(test_en_sentences, translated_sentences)
ter_avg, ter_scores = calculate_ter(test_en_sentences, translated_sentences)

print(f'\n🔹 Overall METEOR Score: {meteor_avg:.4f}')
print(f'🔹 Overall TER Score: {ter_avg:.4f}\n')

[nltk_data] Downloading package wordnet to /aiffel/nltk_data...



🔹 Overall METEOR Score: 0.1045
🔹 Overall TER Score: 111.5018



In [62]:
# ✅ 랜덤 샘플 10개 출력
def print_random_samples(seed=42, num_samples=10):
    set_random_seed(seed)  # 랜덤 시드 고정
    indices = random.sample(range(len(test_ko_sentences)), num_samples)  # 랜덤 인덱스 선택

    print(f'🔹 **Random {num_samples} Sample Translations:** (Seed: {seed})\n')
    for i, idx in enumerate(indices):
        print(f'🔹 Sample {i+1}')
        print(f'  KO: {test_ko_sentences[idx]}')
        print(f'  EN (Reference): {test_en_sentences[idx]}')
        print(f'  EN (Generated): {translated_sentences[idx]}')
        print(f'  METEOR: {meteor_scores[idx]:.4f} | TER: {ter_scores[idx]:.4f}')
        print('-' * 80)
        
# ✅ 랜덤 샘플 출력 실행
print_random_samples(seed=42, num_samples=10)

🔹 **Random 10 Sample Translations:** (Seed: 42)

🔹 Sample 1
  KO: 미국의 정부 과학자들이 초콜릿 공급을 조금 더 안전하게 하기 위해 5년에 걸쳐 실시되는 코코아 열매 게놈 프로젝트에 착수했다.
  EN (Reference): MIAMI, Florida (CNN) U.S. government scientists are launching a five-year project aimed at safeguarding the world's chocolate supply by dissecting the genome of the cocoa bean.
  EN (Generated): the government has unveiled a <unk> of its <unk> as a <unk> of a <unk> for the <unk> to be used to the <unk> of the <unk> .
  METEOR: 0.1627 | TER: 88.4615
--------------------------------------------------------------------------------
🔹 Sample 2
  KO: 올림픽 체제로 전환한 베이징시의 이런 조치가 얼마만큼의 효과를 거둘지는 확신할 수 없다.
  EN (Reference): It is unclear how the effectiveness of the plan will be gauged.
  EN (Generated): the government will not be able to adopt the <unk> <unk> , but it s not clear how to be a visa to the country s most important software .
  METEOR: 0.2174 | TER: 225.0000
---------------------------------------------------------------------------

### 현재 모델의 주요 문제점
OOV (Out-of-Vocabulary) 문제  
단어 사전 크기(20,000)가 부족해 <unk> 토큰이 다수 발생  
특히 고유명사, 기술 용어, 복잡한 단어에서 번역 실패  

문맥 이해 부족  
일부 문장에서 번역된 내용이 원문과 완전히 다름  
예) "온도계가 없다면 어떻게 해야 할까?" → "why does you fix the economy?"  
    
Named Entity(고유명사) 유지 안됨  
"Barack Obama", "Goldwasser" 같은 고유명사가 엉뚱하게 번역됨   
    
### 앞으로 고려해야 할 고도화 방안  
1) OOV 문제 해결  
vocab_size 20,000 → 30,000 이상으로 증가  
SentencePiece 토크나이저 적용 (BPE 방식 사용)  

2) Named Entity(고유명사) 보존  
spaCy NER 적용:  
인명, 지명, 기관명 등을 번역하지 않고 원문 그대로 유지  

3) 데이터 정제 추가 개선  
더 정교한 데이터 필터링  
너무 짧거나 의미 없는 문장 제거  

4) 추가적인 모델 개선 (장기적 목표)  
현재 Seq2Seq + Attention 구조를 유지하지만, 향후 Transformer 적용 고려 가능  
더 좋은 번역 성능을 위해 사전학습된 모델(mBART, MarianMT 등) 파인튜닝 고려 가능  
  
### 최종 결론  
기본적인 Seq2Seq 모델로 학습 및 평가까지 성공적으로 진행됨  
현재 모델은 OOV 문제와 Named Entity 처리에서 개선이 필요함  
SentencePiece 적용과 데이터 정제 강화 후 추가 학습이 필요  

### 회고  
이번 프로젝트를 통해 번역 모델의 성능을 결정짓는 가장 중요한 요소가 데이터 정제와 토큰화 방식이라는 점을 다시금 확인할 수 있었다.   
<unk> 토큰이 과도하게 발생하면서 문장의 의미가 왜곡되는 문제가 발견되었고, 이를 해결하기 위해 vocab_size를 증가시키고 SentencePiece를 활용한 BPE 토크나이저를 적용하면 OOV 문제를 줄일 수 있을것으로 예상된다.   
고유명사처리가 번역 품질에 미치는 영향이 매우 크다는 점을 알게 되었는데,   
의미 없이 변형되거나 누락되면서 문맥이 깨지는 경우가 많았다.   
이를 해결하기 위해 spaCy 기반의 NER 처리를 추가적으로 고려하였고, 향후 적용하면 더욱 개선될 여지가 있다고 판단된다.   
더불어 METEOR와 TER을 함께 분석하면서 모델이 단순히 단어 일치율이 높은 문장을 생성하는 것이 아니라, 문맥적으로 자연스럽고 의미가 통하는 문장을 생성해야 한다는 점을 다시금 깨닫게 되었다.  

## debug
한국어-영어 쌍을 매칭하고, 중복을 제거하며 정제하는 과정에서, 매칭이 뒤틀리고 순서가 뒤죽박죽이 되는 문제 발생  
set 대신 리스트 컴프리헨션으로 변경하니 해결  

In [12]:
import re
from konlpy.tag import Mecab

# Mecab 형태소 분석기 로드
mecab = Mecab()

def preprocess_english(sentence):
    """영어 문장 전처리 함수"""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    sentence = sentence.strip()
    sentence = "<start> " + sentence + " <end>"
    return sentence

def preprocess_korean(sentence):
    """한글 문장 전처리 및 형태소 분석"""
    sentence = sentence.strip()
    sentence = mecab.morphs(sentence)  # 형태소 분석
    sentence = " ".join(sentence)  # 토큰을 공백으로 결합
    return sentence

# 중복 제거
paired_sentences = set(zip(korean_sentences, english_sentences))

# 정제된 데이터 저장
cleaned_corpus = []
for ko, en in paired_sentences:
    ko_cleaned = preprocess_korean(ko.strip())
    en_cleaned = preprocess_english(en.strip())
    cleaned_corpus.append((ko_cleaned, en_cleaned))

# 길이 필터링 (40 토큰 이하)
filtered_corpus = [(ko, en) for ko, en in cleaned_corpus if len(ko.split()) <= 40 and len(en.split()) <= 40]

# 필터링 후 데이터 개수 출력
print(f"정제 후 데이터 개수: {len(filtered_corpus)}")

# 상위 5개 샘플 확인
for i in range(5):
    print(f"KO: {filtered_corpus[i][0]}")
    print(f"EN: {filtered_corpus[i][1]}")
    print()


정제 후 데이터 개수: 59789
KO: 특히 그 는 워터게이트 사건 을 토론 문제 로 삼 았 을 당시 를 언급 했 다 .
EN: <start> particularly , he told cnn , when the topic turned to the watergate scandal . <end>

KO: 온몸 으로 말 하 는 기상 캐스터 온몸 으로 말 하 는 기상 캐스터
EN: <start> their movement on the ball was very good and it shows you much japanese football has improved . <end>

KO: Police guilty over Menezes case 메네제스 사건 에 경찰 과실       2008 . 02
EN: <start> before entering the office building , he said , i m sorry to the people . <end>

KO: 이 들 국가 는 지난해 11 월자 로 비자 면 제국 에 합류 했 다 .
EN: <start> they joined the vwp in november , . <end>

KO: 구조 조정 계획 소식 이 전해 지 면서 모토 로라 의 주식 은 뉴욕 주식 시장 에서 1 % 에 해당 하 는 18 . 46 달러 로 상승 했 다 .
EN: <start> motorola s stock rose to . in after hours trade following the news , up percent from its close of . on the new york stock exchange . <end>



In [13]:
import random

# 랜덤으로 10개 샘플 출력해서 데이터 정렬 확인
random_samples = random.sample(filtered_corpus, 10)

for i, (ko, en) in enumerate(random_samples):
    print(f"{i+1}. KO: {ko}")
    print(f"   EN: {en}")
    print()


1. KO: 미국 산 쇠고기 수입 은 이달 재개 될 예정 이 다 .
   EN: <start> u . s . beef imports are expected to resume this month . <end>

2. KO: 아동 포르노 를 내려 받 은 것 으로 추정 되 는 사람 들 중 아무 도 체포 되 지 않 았 다고 오스트리아 연방 경찰 대변인 이 밝혔 다 .
   EN: <start> none of the people believed to have downloaded the porn have been arrested , said gerald hesztera , a spokesman for austria s federal police service . <end>

3. KO: 보건복지부 김헌주 과장 은 UN 의 성명서 는 구속력 이 없 으며 치료 목적 의 복제 를 허용 하 는 현 정책 을 수정 할 계획 이 없 다고 말 했 다 .
   EN: <start> it is just a non binding declaration and we have no plan to review our policy of allowing therapeutic cloning , the ministry s manager kim heon joo said . <end>

4. KO: 부시 미 대통령 은 “ 이번 암살 주모자 는 반드시 법 의 심판 을 받 아야 한다 ” 며 강력히 규탄 하 고 부토 를 테러 주의자 들 과 맞선 여성 지도자 로 칭송 했 다 .
   EN: <start> president bush said those responsible must be brought to justice and praised bhutto as a woman who had fought the forces of terror . <end>

5. KO: 고래 의 나이 를 계산 하 기 는 어려울 수 있 으며 보통 고래 의 나이 는 수정체 에 있 는 아미노산 을 통해 판단 한다 .
   EN: <start> 